# Phase 4 v2: Model Training & Evaluation

## Objective

Train and evaluate machine learning models on the optimized feature set from Phase 3.

**Input**: Phase 3 optimized dataset (283,118 records, 56 columns, 49 features)  
**Output**: Trained models with performance metrics 

**Dataset Characteristics:**
- **Total records**: 283,118
- **Unique cases**: 18 (all contain timestomping events)
- **Timestomped events**: 280 (0.10%)
- **Normal events**: 282,838 (99.90%)
- **Extreme imbalance ratio**: 1:1,010

---

## Models to Train

1. **Random Forest** - Baseline ensemble model
2. **XGBoost** - Gradient boosting for imbalanced data
3. **LightGBM** - Fast gradient boosting
4. **Logistic Regression** - Interpretable linear model

---

## Evaluation Strategy

### Forensically-Relevant Metrics (Priority Order)

**Primary Metrics:**
1. **F1-Score** - Best single metric for imbalanced data; balances precision and recall
2. **Recall (Detection Rate)** - *"Of all actual attacks, how many did we catch?"*
   - **Most critical for forensics** - Missing attacks is worse than false alarms
3. **Precision** - *"Of flagged events, how many are real attacks?"*
   - **Impacts analyst workload** - Low precision = wasted investigation time

**Error Analysis:**
4. **False Negative Rate (FNR)** - *"What % of attacks did we miss?"* (Security risk metric)
5. **False Positive Rate (FPR)** - *"What % of normal events flagged as attacks?"* (Workload metric)

**Reference Metrics (less meaningful with extreme imbalance):**
- **Accuracy** - A naive "predict all normal" model achieves 99.9% accuracy
- **AUC-ROC** - Good for model comparison but not interpretable in practice

### Train-Test Split Strategy
- **Case-aware splitting** - Ensure same case_id doesn't appear in both train/test (prevents data leakage)
- **Stratified by case** - Maintain timestomping distribution across train/test
- **80/20 split**: 14 cases train, 4 cases test
- **random_state=42** for reproducibility

---

## Key Considerations

### Forensic Context
- **Post-incident investigation** - Analyst already reviewing the case (not real-time detection)
- **High missing data** (98-99%) in forensic columns is expected (artifact sparsity)
- **Low-variance features** (has_attribute_change, etc.) detect rare but critical APT techniques
- **Workload reduction goal** - Reduce analyst search space from thousands to dozens of events

### Class Imbalance Handling
- **Random Forest / Logistic Regression**: `class_weight='balanced'`
- **XGBoost / LightGBM**: `scale_pos_weight` parameter (ratio of negative to positive class)

---

## 1. Setup & Load Data

In [177]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Scikit-learn imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, roc_curve, f1_score, 
    precision_score, recall_score, accuracy_score
)

# XGBoost and LightGBM
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")

Libraries imported successfully


In [178]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 3 - V2 Feature Selection'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 4 - V2 Model Training'
MODEL_DIR = BASE_DIR / 'models' / 'v2'

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Model version for retraining with drop_first=False
MODEL_VERSION = "v3"  # v2 used drop_first=True, v3 uses drop_first=False for booleans

print("Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Models: {MODEL_DIR}")
print(f"  Model Version: {MODEL_VERSION} (with drop_first=False for boolean columns)")

Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - V2 Model Training
  Models: /Users/soni/Github/Digital-Detectives_Thesis/models/v2
  Model Version: v3 (with drop_first=False for boolean columns)


In [179]:
# Load Phase 3 optimized dataset
print("Loading Phase 3 optimized dataset...")
input_file = INPUT_DIR / 'all_cases_combined_v2_phase3_final.csv'

# Force case_id to string to prevent pandas from auto-converting to int
df = pd.read_csv(input_file, encoding='utf-8-sig', dtype={'case_id': str})

print(f"\nDataset loaded successfully:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Case IDs: {df['case_id'].nunique()} unique (type: {df['case_id'].dtype})")
print(f"  Timestomped events: {(df['timestomped'] == 1).sum():,} ({(df['timestomped'] == 1).sum() / len(df) * 100:.2f}%)")
print(f"  Normal events: {(df['timestomped'] == 0).sum():,} ({(df['timestomped'] == 0).sum() / len(df) * 100:.2f}%)")
print(f"  Imbalance ratio: 1:{int((df['timestomped'] == 0).sum() / (df['timestomped'] == 1).sum())}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Loading Phase 3 optimized dataset...

Dataset loaded successfully:
  Records: 283,118
  Columns: 56
  Case IDs: 18 unique (type: object)
  Timestomped events: 280 (0.10%)
  Normal events: 282,838 (99.90%)
  Imbalance ratio: 1:1010
  Memory usage: 412.42 MB


---
## 2. Feature Preparation

In [180]:
print("=" * 80)
print("FEATURE PREPARATION")
print("=" * 80)

# Define column categories
identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']
target_col = 'timestomped'
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

print(f"\nColumn breakdown:")
print(f"  Identifiers: {len(identifier_cols)}")
print(f"  Target: 1")
print(f"  Features: {len(feature_cols)}")

# Separate numeric and non-numeric features
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_features = [col for col in feature_cols if col not in numeric_features]

print(f"\nFeature types:")
print(f"  Numeric: {len(numeric_features)}")
print(f"  Non-numeric: {len(non_numeric_features)}")

if non_numeric_features:
    print(f"\nNon-numeric features that need encoding:")
    for col in non_numeric_features:
        unique_vals = df[col].nunique()
        missing_pct = df[col].isnull().sum() / len(df) * 100
        print(f"  - {col:45s}: {unique_vals:5d} unique values, {missing_pct:5.1f}% missing")

FEATURE PREPARATION

Column breakdown:
  Identifiers: 6
  Target: 1
  Features: 49

Feature types:
  Numeric: 18
  Non-numeric: 31

Non-numeric features that need encoding:
  - lf_event                                     :     3 unique values,  98.3% missing
  - lf_target_vcn                                :  1327 unique values,  98.3% missing
  - usn_event_info                               :   102 unique values,   0.2% missing
  - usn_file_reference_number                    : 31647 unique values,   0.2% missing
  - usn_parent_file_reference_number             :  5229 unique values,   0.2% missing
  - source                                       :     3 unique values,   0.0% missing
  - is_tunneling                                 :     2 unique values,   0.0% missing
  - lf_creation_time_before                      :   520 unique values,  99.5% missing
  - lf_creation_time_after                       :    95 unique values,  99.5% missing
  - lf_modified_time_before                 

---
## 3. Data Preprocessing Strategy

We need to handle:
1. **Non-numeric features** - Encode categorical variables
2. **Missing data** - Strategy for forensic columns with high missingness
3. **Class imbalance** - Use class weights or sampling techniques

### Encoding Strategy
- **Boolean columns** - Convert True/False to 1/0
- **Low-cardinality strings** (< 10 unique) - One-hot encoding
- **High-cardinality strings** (>= 10 unique) - Label encoding or drop if not useful
- **Datetime strings** - Drop (already have derived temporal features)

### Missing Data Strategy
- **Forensic columns** (lf_*, 98-99% missing) - Keep as-is, fill with -1 or 'missing'
- **Other columns** - Fill with appropriate defaults

Let's create a preprocessing pipeline.

In [181]:
print("=" * 80)
print("DATA PREPROCESSING")
print("=" * 80)

# Create a copy for preprocessing
df_processed = df.copy()

# Separate X and y
X = df_processed[feature_cols].copy()
y = df_processed[target_col].copy()

print(f"\nInitial shape: X={X.shape}, y={y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

# Store original feature names for later analysis
original_features = X.columns.tolist()

DATA PREPROCESSING

Initial shape: X=(283118, 49), y=(283118,)

Target distribution:
timestomped
0    282838
1       280
Name: count, dtype: int64


In [182]:
print("\n" + "=" * 80)
print("STEP 1: Handle Boolean Columns")
print("=" * 80)

# Convert boolean columns to int (True/False -> 1/0)
bool_cols = X.select_dtypes(include=['bool']).columns.tolist()

if bool_cols:
    print(f"\nConverting {len(bool_cols)} boolean columns to int:")
    for col in bool_cols:
        X[col] = X[col].astype(int)
        print(f"  ✓ {col}")
else:
    print("\nNo boolean columns found")


STEP 1: Handle Boolean Columns

Converting 12 boolean columns to int:
  ✓ is_tunneling
  ✓ zero_in_nanoseconds
  ✓ is_executable
  ✓ is_system_file
  ✓ is_hidden_file
  ✓ is_archive
  ✓ has_suspicious_extension
  ✓ has_logfile_evidence
  ✓ has_usnjrnl_evidence
  ✓ usn_basic_info_change
  ✓ usn_file_closed
  ✓ usn_complete_manipulation_pattern


In [183]:
print("\n" + "=" * 80)
print("STEP 2: Handle Object/String Columns")
print("=" * 80)

# Get object columns
object_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nFound {len(object_cols)} object columns")

# Analyze each object column
cols_to_drop = []
cols_to_encode = []

for col in object_cols:
    unique_vals = X[col].nunique()
    missing_pct = X[col].isnull().sum() / len(X) * 100
    
    print(f"\n{col}:")
    print(f"  Unique values: {unique_vals}")
    print(f"  Missing: {missing_pct:.1f}%")
    
    # Decision logic
    if unique_vals > 1000:  # High cardinality - likely identifiers
        print(f"  -> DROP (high cardinality, likely identifier)")
        cols_to_drop.append(col)
    elif unique_vals < 10:  # Low cardinality - can one-hot encode
        print(f"  -> ENCODE (low cardinality)")
        cols_to_encode.append(col)
    else:  # Medium cardinality - label encode
        print(f"  -> LABEL ENCODE (medium cardinality)")
        cols_to_encode.append(col)

print(f"\n" + "=" * 80)
print(f"Summary: Drop {len(cols_to_drop)}, Encode {len(cols_to_encode)}")
print("=" * 80)


STEP 2: Handle Object/String Columns

Found 19 object columns

lf_event:
  Unique values: 3
  Missing: 98.3%
  -> ENCODE (low cardinality)

lf_target_vcn:
  Unique values: 1327
  Missing: 98.3%
  -> DROP (high cardinality, likely identifier)

usn_event_info:
  Unique values: 102
  Missing: 0.2%
  -> LABEL ENCODE (medium cardinality)

usn_file_reference_number:
  Unique values: 31647
  Missing: 0.2%
  -> DROP (high cardinality, likely identifier)

usn_parent_file_reference_number:
  Unique values: 5229
  Missing: 0.2%
  -> DROP (high cardinality, likely identifier)

source:
  Unique values: 3
  Missing: 0.0%
  -> ENCODE (low cardinality)

lf_creation_time_before:
  Unique values: 520
  Missing: 99.5%
  -> LABEL ENCODE (medium cardinality)

lf_creation_time_after:
  Unique values: 95
  Missing: 99.5%
  -> LABEL ENCODE (medium cardinality)

lf_modified_time_before:
  Unique values: 240
  Missing: 98.7%
  -> LABEL ENCODE (medium cardinality)

lf_modified_time_after:
  Unique values: 330
 

In [184]:
print("\n" + "=" * 80)
print("STEP 3: Drop High-Cardinality Columns")
print("=" * 80)

if cols_to_drop:
    print(f"\nDropping {len(cols_to_drop)} columns:")
    for col in cols_to_drop:
        print(f"  - {col}")
    X = X.drop(columns=cols_to_drop)
    print(f"\nNew shape: {X.shape}")
else:
    print("\nNo columns to drop")


STEP 3: Drop High-Cardinality Columns

Dropping 3 columns:
  - lf_target_vcn
  - usn_file_reference_number
  - usn_parent_file_reference_number

New shape: (283118, 46)


In [185]:
print("\n" + "=" * 80)
print("STEP 4: Encode Categorical Columns")
print("=" * 80)

# For each column to encode, decide between one-hot and label encoding
label_encoders = {}

# Boolean-like columns that need all three states preserved (True, False, missing)
boolean_like_cols = ['copied_from_file', 'creation_time_changed_to_past', 'modified_time_changed_to_past', 
                     'accessed_time_changed_to_past', 'mft_modified_time_changed_to_past']

for col in cols_to_encode:
    if col not in X.columns:
        continue
        
    unique_vals = X[col].nunique()
    
    if unique_vals < 5:  # One-hot encode
        # Fill missing values with 'missing' before encoding
        X[col] = X[col].fillna('missing')
        
        # Use drop_first=False for boolean-like columns to preserve all three states
        if col in boolean_like_cols:
            print(f"\n{col}: One-hot encoding ({unique_vals} categories, drop_first=False for boolean)")
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=False)
            print(f"  Created {len(dummies.columns)} dummy columns: {', '.join(dummies.columns.tolist())}")
        else:
            print(f"\n{col}: One-hot encoding ({unique_vals} categories)")
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
            print(f"  Created {len(dummies.columns)} dummy columns")
        
        X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
    else:  # Label encode
        print(f"\n{col}: Label encoding ({unique_vals} categories)")
        # Fill missing values with 'missing' before encoding
        X[col] = X[col].fillna('missing')
        # Label encode
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        label_encoders[col] = le
        print(f"  Encoded to integers 0-{unique_vals-1}")

print(f"\nFinal shape after encoding: {X.shape}")


STEP 4: Encode Categorical Columns

lf_event: One-hot encoding (3 categories)
  Created 3 dummy columns

usn_event_info: Label encoding (102 categories)
  Encoded to integers 0-101

source: One-hot encoding (3 categories)
  Created 2 dummy columns

lf_creation_time_before: Label encoding (520 categories)
  Encoded to integers 0-519

lf_creation_time_after: Label encoding (95 categories)
  Encoded to integers 0-94

lf_modified_time_before: Label encoding (240 categories)
  Encoded to integers 0-239

lf_modified_time_after: Label encoding (330 categories)
  Encoded to integers 0-329

lf_accessed_time_before: Label encoding (102 categories)
  Encoded to integers 0-101

lf_accessed_time_after: Label encoding (48 categories)
  Encoded to integers 0-47

lf_mft_modified_time_before: Label encoding (159 categories)
  Encoded to integers 0-158

lf_mft_modified_time_after: Label encoding (186 categories)
  Encoded to integers 0-185

copied_from_file: One-hot encoding (2 categories, drop_first=F

In [186]:
print("\n" + "=" * 80)
print("STEP 5: Handle Missing Values in Numeric Columns")
print("=" * 80)

# Check for remaining missing values
missing_cols = X.columns[X.isnull().any()].tolist()

if missing_cols:
    print(f"\nFound {len(missing_cols)} columns with missing values:")
    for col in missing_cols:
        missing_count = X[col].isnull().sum()
        missing_pct = missing_count / len(X) * 100
        print(f"  {col:45s}: {missing_count:7,} ({missing_pct:5.1f}%)")
    
    print(f"\nFilling missing values with -1 (forensic 'not present' indicator)")
    X = X.fillna(-1)
    print(f"✓ Missing values handled")
else:
    print("\nNo missing values found")

print(f"\nFinal preprocessed shape: X={X.shape}, y={y.shape}")


STEP 5: Handle Missing Values in Numeric Columns

Found 6 columns with missing values:
  lf_lsn                                       : 278,393 ( 98.3%)
  lf_cluster_index                             : 278,393 ( 98.3%)
  usn_usn                                      :     531 (  0.2%)
  accessed_time_delta_days                     : 282,826 ( 99.9%)
  mft_modified_time_delta_days                 : 281,649 ( 99.5%)
  event_vs_modified_after_days                 : 280,390 ( 99.0%)

Filling missing values with -1 (forensic 'not present' indicator)


✓ Missing values handled

Final preprocessed shape: X=(283118, 59), y=(283118,)


In [187]:
print("\n" + "=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)

print(f"\nOriginal features: {len(original_features)}")
print(f"Final features: {len(X.columns)}")
print(f"Features added/removed: {len(X.columns) - len(original_features):+d}")

print(f"\nData types:")
print(X.dtypes.value_counts())

print(f"\nMemory usage: {X.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


PREPROCESSING SUMMARY

Original features: 49
Final features: 59
Features added/removed: +10

Data types:
int64      31
bool       20
float64     8
Name: count, dtype: int64

Memory usage: 89.64 MB


---
## 4. Train-Test Split

### Case-Aware Splitting
We need to ensure that all events from the same case stay together in either train or test set.
This prevents data leakage where the model learns case-specific patterns.

In [188]:
print("=" * 80)
print("TRAIN-TEST SPLIT (CASE-AWARE)")
print("=" * 80)

# Get case IDs from original dataframe
case_ids = df['case_id'].values

# Get unique cases and their labels
unique_cases = df[['case_id', 'timestomped']].groupby('case_id')['timestomped'].max().reset_index()
print(f"\nTotal unique cases: {len(unique_cases)}")
print(f"Cases with timestomping: {(unique_cases['timestomped'] == 1).sum()}")
print(f"Cases without timestomping: {(unique_cases['timestomped'] == 0).sum()}")

# Split cases (not individual records)
train_cases, test_cases = train_test_split(
    unique_cases['case_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=unique_cases['timestomped'].values
)

print(f"\nTrain cases: {len(train_cases)}")
print(f"Test cases: {len(test_cases)}")

# Create train and test masks
train_mask = df['case_id'].isin(train_cases)
test_mask = df['case_id'].isin(test_cases)

# Split data
X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\nTrain set: {X_train.shape[0]:,} records ({(y_train == 1).sum()} timestomped)")
print(f"Test set: {X_test.shape[0]:,} records ({(y_test == 1).sum()} timestomped)")

print(f"\nClass distribution:")
print(f"  Train: {(y_train == 1).sum() / len(y_train) * 100:.2f}% timestomped")
print(f"  Test: {(y_test == 1).sum() / len(y_test) * 100:.2f}% timestomped")

TRAIN-TEST SPLIT (CASE-AWARE)

Total unique cases: 18
Cases with timestomping: 18
Cases without timestomping: 0

Train cases: 14
Test cases: 4

Train set: 238,108 records (202 timestomped)
Test set: 45,010 records (78 timestomped)

Class distribution:
  Train: 0.08% timestomped
  Test: 0.17% timestomped


---
## 5. Model Training

We'll train 4 models with class weight balancing to handle the extreme imbalance.

In [189]:
print("=" * 80)
print("MODEL TRAINING")
print("=" * 80)

# Store models and results
models = {}
results = {}

# Calculate scale_pos_weight for XGBoost/LightGBM
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nScale pos weight (for XGBoost/LightGBM): {scale_pos_weight:.2f}")

MODEL TRAINING

Scale pos weight (for XGBoost/LightGBM): 1177.75


In [190]:
print("\n" + "=" * 80)
print("MODEL 1: Random Forest")
print("=" * 80)

print("\nTraining Random Forest with class_weight='balanced'...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model

print("\n✓ Random Forest training complete")


MODEL 1: Random Forest

Training Random Forest with class_weight='balanced'...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    1.0s



✓ Random Forest training complete


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    1.8s finished


In [191]:
print("\n" + "=" * 80)
print("MODEL 2: XGBoost")
print("=" * 80)

print("\nTraining XGBoost with scale_pos_weight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train, verbose=False)
models['XGBoost'] = xgb_model

print("\n✓ XGBoost training complete")


MODEL 2: XGBoost

Training XGBoost with scale_pos_weight...

✓ XGBoost training complete


In [192]:
print("\n" + "=" * 80)
print("MODEL 3: LightGBM")
print("=" * 80)

print("\nTraining LightGBM with scale_pos_weight...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_train, y_train)
models['LightGBM'] = lgb_model

print("\n✓ LightGBM training complete")


MODEL 3: LightGBM

Training LightGBM with scale_pos_weight...

✓ LightGBM training complete


In [193]:
print("\n" + "=" * 80)
print("MODEL 4: Logistic Regression")
print("=" * 80)

print("\nTraining Logistic Regression with class_weight='balanced'...")
# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr_model

print("\n✓ Logistic Regression training complete")


MODEL 4: Logistic Regression

Training Logistic Regression with class_weight='balanced'...

✓ Logistic Regression training complete


---
## 6. Model Evaluation

### Metrics for Imbalanced Data

With extreme class imbalance (1:1,010 ratio), standard metrics like **Accuracy** and **AUC-ROC** are misleading:

- **Accuracy** - A naive "predict all normal" model achieves 99.9% accuracy
- **AUC-ROC** - Good for model comparison but not interpretable in practice

### Forensically-Relevant Metrics

For timestomping detection, we prioritize:

1. **Recall (Sensitivity)** - *"Of all actual attacks, how many did we catch?"*
   - **Most critical for forensics** - Missing attacks is worse than false alarms
   - Goal: Maximize recall (catch as many attacks as possible)

2. **Precision** - *"Of flagged events, how many are real attacks?"*
   - **Impacts analyst workload** - Low precision = wasted investigation time
   - Goal: Balance precision with recall (minimize false alarms)

3. **F1-Score** - *Harmonic mean of precision and recall*
   - **Best single metric** for imbalanced data
   - Goal: Optimize F1-score for best overall balance

4. **False Negative Rate (FNR)** - *"What % of attacks did we miss?"*
   - **Security risk metric** - Lower is better
   - Calculated as: FN / (FN + TP)

5. **False Positive Rate (FPR)** - *"What % of normal events flagged as attacks?"*
   - **Analyst workload metric** - Lower is better
   - Calculated as: FP / (FP + TN)

### Evaluation Strategy

We report metrics in order of forensic importance:
- Primary: **Recall, Precision, F1-Score**
- Secondary: **FNR, FPR**
- Reference only: Accuracy, AUC-ROC

In [194]:
print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

# Evaluate each model
for model_name, model in models.items():
    print(f"\n" + "=" * 80)
    print(f"{model_name.upper()}")
    print("=" * 80)
    
    # Get predictions
    if model_name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    try:
        auc_roc = roc_auc_score(y_test, y_pred_proba)
    except:
        auc_roc = 0.0
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Calculate FNR and FPR
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'fnr': fnr,
        'fpr': fpr,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    # Print forensically-relevant metrics FIRST
    print(f"\nFORENSIC PERFORMANCE METRICS:")
    print(f"  Recall (Detection Rate):     {recall:.4f} ({recall*100:.2f}%)")
    print(f"  Precision (Accuracy of Flags): {precision:.4f} ({precision*100:.2f}%)")
    print(f"  F1-Score (Overall Balance):   {f1:.4f}")
    
    print(f"\nERROR ANALYSIS:")
    print(f"  False Negative Rate:  {fnr:.4f} ({fnr*100:.2f}%) - Missed {fn}/{fn+tp} attacks")
    print(f"  False Positive Rate:  {fpr:.4f} ({fpr*100:.2f}%) - {fp} false alarms per {fp+tn:,} normal events")
    
    # Confusion matrix
    print(f"\nCONFUSION MATRIX:")
    print(f"                  Predicted")
    print(f"                  Normal    Attack")
    print(f"  Actual Normal   {tn:6,}    {fp:6,}   ({tn/(tn+fp)*100:.2f}% correct)")
    print(f"  Actual Attack   {fn:6,}    {tp:6,}   ({tp/(tp+fn)*100:.2f}% detected)")
    
    # Reference metrics (de-emphasized)
    print(f"\nREFERENCE METRICS:")
    print(f"  Accuracy:  {accuracy:.4f} (less meaningful with imbalance)")
    print(f"  AUC-ROC:   {auc_roc:.4f} (good for comparison)")
    
    # Practical interpretation
    print(f"\nPRACTICAL INTERPRETATION:")
    total_attacks = tp + fn
    detected = tp
    missed = fn
    false_alarms = fp
    
    print(f"  • Detected {detected}/{total_attacks} attacks ({recall*100:.1f}%)")
    print(f"  • Missed {missed}/{total_attacks} attacks ({fnr*100:.1f}%)")
    print(f"  • Generated {false_alarms} false alarms ({fpr*100:.3f}% of normal events)")
    
    if fp > 0:
        ppv = tp / (tp + fp)  # Positive Predictive Value
        print(f"  • Of {tp+fp} flagged events, {tp} were real attacks ({ppv*100:.1f}% precision)")
        print(f"  • Analyst would investigate {fp} false alarms per {total_attacks} real attacks")

MODEL EVALUATION

RANDOM FOREST


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s



FORENSIC PERFORMANCE METRICS:
  Recall (Detection Rate):     0.9231 (92.31%)
  Precision (Accuracy of Flags): 0.6102 (61.02%)
  F1-Score (Overall Balance):   0.7347

ERROR ANALYSIS:
  False Negative Rate:  0.0769 (7.69%) - Missed 6/78 attacks
  False Positive Rate:  0.0010 (0.10%) - 46 false alarms per 44,932 normal events

CONFUSION MATRIX:
                  Predicted
                  Normal    Attack
  Actual Normal   44,886        46   (99.90% correct)
  Actual Attack        6        72   (92.31% detected)

REFERENCE METRICS:
  Accuracy:  0.9988 (less meaningful with imbalance)
  AUC-ROC:   0.9995 (good for comparison)

PRACTICAL INTERPRETATION:
  • Detected 72/78 attacks (92.3%)
  • Missed 6/78 attacks (7.7%)
  • Generated 46 false alarms (0.102% of normal events)
  • Of 118 flagged events, 72 were real attacks (61.0% precision)
  • Analyst would investigate 46 false alarms per 78 real attacks

XGBOOST

FORENSIC PERFORMANCE METRICS:
  Recall (Detection Rate):     0.0513 (5.13%)
 

[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.2s finished


---
## 7. Model Comparison

In [195]:
print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)

# Create comparison dataframe with forensically-relevant metrics
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'FNR': [results[m]['fnr'] for m in results.keys()],
    'FPR': [results[m]['fpr'] for m in results.keys()],
    'Attacks Detected': [f"{results[m]['tp']}/{results[m]['tp']+results[m]['fn']}" for m in results.keys()],
    'False Alarms': [results[m]['fp'] for m in results.keys()],
    'AUC-ROC': [results[m]['auc_roc'] for m in results.keys()],
    'Accuracy': [results[m]['accuracy'] for m in results.keys()]
})

# Sort by F1-Score (best overall balance)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\nFORENSIC PERFORMANCE COMPARISON")
print("=" * 80)
print("\nPrimary Metrics (ordered by F1-Score):")
print(comparison_df[['Model', 'F1-Score', 'Recall', 'Precision', 'Attacks Detected', 'False Alarms']].to_string(index=False))

print("\n\nError Rates:")
print(comparison_df[['Model', 'FNR', 'FPR']].to_string(index=False))

print("\n\nReference Metrics (less meaningful with imbalance):")
print(comparison_df[['Model', 'Accuracy', 'AUC-ROC']].to_string(index=False))

# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
best_row = comparison_df.iloc[0]

print(f"\n" + "=" * 80)
print(f"BEST MODEL: {best_model_name}")
print("=" * 80)
print(f"\nForensic Performance:")
print(f"  F1-Score:  {best_row['F1-Score']:.4f}")
print(f"  Recall:    {best_row['Recall']:.4f} ({best_row['Recall']*100:.2f}% of attacks detected)")
print(f"  Precision: {best_row['Precision']:.4f} ({best_row['Precision']*100:.2f}% of flags are real)")
print(f"  FNR:       {best_row['FNR']:.4f} ({best_row['FNR']*100:.2f}% of attacks missed)")
print(f"  FPR:       {best_row['FPR']:.4f} ({best_row['FPR']*100:.3f}% false alarm rate)")

print(f"\nPractical Summary:")
print(f"  ✓ Detected {best_row['Attacks Detected']} timestomped events")
print(f"  ✓ Generated {int(best_row['False Alarms'])} false alarms")

# Calculate workload ratio
tp = results[best_model_name]['tp']
fp = results[best_model_name]['fp']
workload_ratio = fp / tp if tp > 0 else 0
print(f"  ✓ Analyst investigates {workload_ratio:.2f} false alarms per real attack")

print("\n" + "=" * 80)
print("INTERPRETATION FOR FORENSIC USE CASE:")
print("=" * 80)

if best_row['Recall'] >= 0.85:
    print(f"✓ EXCELLENT Detection: Catches {best_row['Recall']*100:.1f}% of attacks")
elif best_row['Recall'] >= 0.70:
    print(f"✓ GOOD Detection: Catches {best_row['Recall']*100:.1f}% of attacks")
else:
    print(f"MODERATE Detection: Catches {best_row['Recall']*100:.1f}% of attacks (improvement needed)")

if best_row['Precision'] >= 0.70:
    print(f"✓ LOW False Alarm Rate: {best_row['Precision']*100:.1f}% of flagged events are real attacks")
elif best_row['Precision'] >= 0.50:
    print(f"MODERATE False Alarm Rate: {best_row['Precision']*100:.1f}% of flagged events are real attacks")
else:
    print(f"HIGH False Alarm Rate: {best_row['Precision']*100:.1f}% of flagged events are real attacks")

if workload_ratio <= 1.0:
    print(f"✓ EFFICIENT: For every real attack, analyst investigates {workload_ratio:.1f} false alarms")
elif workload_ratio <= 3.0:
    print(f"ACCEPTABLE: For every real attack, analyst investigates {workload_ratio:.1f} false alarms")
else:
    print(f"HIGH WORKLOAD: For every real attack, analyst investigates {workload_ratio:.1f} false alarms")

MODEL COMPARISON

FORENSIC PERFORMANCE COMPARISON

Primary Metrics (ordered by F1-Score):
              Model  F1-Score   Recall  Precision Attacks Detected  False Alarms
      Random Forest  0.734694 0.923077   0.610169            72/78            46
           LightGBM  0.240575 0.858974   0.139875            67/78           412
            XGBoost  0.093023 0.051282   0.500000             4/78             4
Logistic Regression  0.059342 0.948718   0.030629            74/78          2342


Error Rates:
              Model      FNR      FPR
      Random Forest 0.076923 0.001024
           LightGBM 0.141026 0.009169
            XGBoost 0.948718 0.000089
Logistic Regression 0.051282 0.052123


Reference Metrics (less meaningful with imbalance):
              Model  Accuracy  AUC-ROC
      Random Forest  0.998845 0.999547
           LightGBM  0.990602 0.924896
            XGBoost  0.998267 0.999290
Logistic Regression  0.947878 0.976279

BEST MODEL: Random Forest

Forensic Performance:
 

---
## 8. Feature Importance Analysis

Analyze which features are most important for the best-performing models.

In [196]:
print("=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# Get feature importance from tree-based models
tree_models = ['Random Forest', 'XGBoost', 'LightGBM']

for model_name in tree_models:
    if model_name in models:
        print(f"\n" + "=" * 80)
        print(f"{model_name} - Top 20 Features")
        print("=" * 80)
        
        model = models[model_name]
        
        # Get feature importance
        if hasattr(model, 'feature_importances_'):
            importance = model.feature_importances_
            feature_importance_df = pd.DataFrame({
                'Feature': X_train.columns,
                'Importance': importance
            }).sort_values('Importance', ascending=False)
            
            print("\n" + feature_importance_df.head(20).to_string(index=False))

FEATURE IMPORTANCE ANALYSIS

Random Forest - Top 20 Features

                             Feature  Importance
            event_frequency_per_file    0.213724
                             usn_usn    0.156030
                      usn_event_info    0.097981
                          path_depth    0.076017
               events_in_5min_window    0.074966
                     filename_length    0.047193
timestamp_manipulation_pattern_score    0.046843
                          is_archive    0.038685
            event_frequency_per_case    0.027955
   usn_complete_manipulation_pattern    0.022798
                     usn_file_closed    0.018539
                       is_executable    0.016881
                              lf_lsn    0.014271
     cross_artifact_validation_score    0.012024
                    lf_cluster_index    0.011355
              copied_from_file_False    0.009861
              lf_creation_time_after    0.009755
                 source_usnjrnl_only    0.008939
       

---
## 9. Save Models and Results

In [197]:
print("=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import json

# Save models
for model_name, model in models.items():
    model_file = MODEL_DIR / f"{model_name.lower().replace(' ', '_')}_v2.pkl"
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Saved {model_name} to {model_file}")

# Save scaler (for logistic regression)
scaler_file = MODEL_DIR / 'scaler_v2.pkl'
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Saved scaler to {scaler_file}")

# Save comparison results
results_file = OUTPUT_DIR / 'model_comparison_v2.csv'
comparison_df.to_csv(results_file, index=False)
print(f"✓ Saved comparison results to {results_file}")

# Save feature names
feature_names_file = OUTPUT_DIR / 'feature_names_v2.json'
with open(feature_names_file, 'w') as f:
    json.dump(X_train.columns.tolist(), f, indent=2)
print(f"✓ Saved feature names to {feature_names_file}")

SAVING MODELS AND RESULTS
✓ Saved Random Forest to /Users/soni/Github/Digital-Detectives_Thesis/models/v2/random_forest_v2.pkl
✓ Saved XGBoost to /Users/soni/Github/Digital-Detectives_Thesis/models/v2/xgboost_v2.pkl
✓ Saved LightGBM to /Users/soni/Github/Digital-Detectives_Thesis/models/v2/lightgbm_v2.pkl
✓ Saved Logistic Regression to /Users/soni/Github/Digital-Detectives_Thesis/models/v2/logistic_regression_v2.pkl
✓ Saved scaler to /Users/soni/Github/Digital-Detectives_Thesis/models/v2/scaler_v2.pkl
✓ Saved comparison results to /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - V2 Model Training/model_comparison_v2.csv
✓ Saved feature names to /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - V2 Model Training/feature_names_v2.json


---
## 10. Summary Report

In [198]:
print("\n" + "=" * 80)
print("PHASE 4 v2.0 SUMMARY REPORT - MODEL TRAINING")
print("=" * 80)

print("\nPHASE 4 v2.0 COMPLETE - MODEL TRAINING & EVALUATION")

print("\n" + "=" * 80)
print("1. DATASET")
print("=" * 80)
print(f"  Total records: {len(df):,}")
print(f"  Features: {len(X.columns)}")
print(f"  Train set: {len(X_train):,} records")
print(f"  Test set: {len(X_test):,} records")
print(f"  Class imbalance ratio: 1:{int(scale_pos_weight)}")

print("\n" + "=" * 80)
print("2. MODELS TRAINED")
print("=" * 80)
for i, model_name in enumerate(models.keys(), 1):
    print(f"  {i}. {model_name}")

print("\n" + "=" * 80)
print("3. BEST MODEL")
print("=" * 80)
best_row = comparison_df.iloc[0]
print(f"  Model: {best_row['Model']}")
print(f"  F1-Score: {best_row['F1-Score']:.4f}")
print(f"  Precision: {best_row['Precision']:.4f}")
print(f"  Recall: {best_row['Recall']:.4f}")

print("\n" + "=" * 80)
print("4. MODEL COMPARISON")
print("=" * 80)
print("\n" + comparison_df.to_string(index=False))

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("\n1. Analyze feature importance in detail")
print("2. Perform error analysis on false positives/negatives")
print("3. Consider hyperparameter tuning for best model")
print("4. Compare with Phase 1 baseline performance")
print("5. Generate final thesis results and visualizations")

print("\n" + "=" * 80)
print("OUTPUT FILES")
print("=" * 80)
print(f"  Models saved to: {MODEL_DIR}")
print(f"  Results saved to: {OUTPUT_DIR}")
print("=" * 80)


PHASE 4 v2.0 SUMMARY REPORT - MODEL TRAINING

PHASE 4 v2.0 COMPLETE - MODEL TRAINING & EVALUATION

1. DATASET
  Total records: 283,118
  Features: 59
  Train set: 238,108 records
  Test set: 45,010 records
  Class imbalance ratio: 1:1177

2. MODELS TRAINED
  1. Random Forest
  2. XGBoost
  3. LightGBM
  4. Logistic Regression

3. BEST MODEL
  Model: Random Forest
  F1-Score: 0.7347
  Precision: 0.6102
  Recall: 0.9231

4. MODEL COMPARISON

              Model  F1-Score   Recall  Precision      FNR      FPR Attacks Detected  False Alarms  AUC-ROC  Accuracy
      Random Forest  0.734694 0.923077   0.610169 0.076923 0.001024            72/78            46 0.999547  0.998845
           LightGBM  0.240575 0.858974   0.139875 0.141026 0.009169            67/78           412 0.924896  0.990602
            XGBoost  0.093023 0.051282   0.500000 0.948718 0.000089             4/78             4 0.999290  0.998267
Logistic Regression  0.059342 0.948718   0.030629 0.051282 0.052123            74/7